In [ ]:
import pandas as pd

df = pd.read_csv('unified_dataset.csv')

PATHOLOGY_COLS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Pleural_Effusion', 'Pneumonia', 'Pneumothorax', 'Infiltration',
    'Nodule_Mass', 'Pleural_Thickening', 'No_Finding'
]

print(f'Total images: {len(df):,}')
df.head(3)

## 1. Imágenes por patología (valor == 1)

In [ ]:
counts = (
    (df[PATHOLOGY_COLS] == 1)
    .sum()
    .rename('positive_images')
    .sort_values(ascending=False)
    .to_frame()
)
counts['pct_of_total'] = (counts['positive_images'] / len(df) * 100).round(2)
counts

## 2. Imágenes con múltiples patologías confirmadas (valor == 1)

In [ ]:
# Exclude No_Finding from multi-label count
disease_cols = [c for c in PATHOLOGY_COLS if c != 'No_Finding']

df['n_positive'] = (df[disease_cols] == 1).sum(axis=1)

multi = df[df['n_positive'] > 1]
print(f'Images with 2+ confirmed pathologies: {len(multi):,} ({len(multi)/len(df)*100:.2f}%)')

print('\nDistribution of co-occurring pathology count:')
print(df['n_positive'].value_counts().sort_index().to_string())

print('\nSample rows with multiple confirmed pathologies:')
multi[['image_id', 'source', 'n_positive'] + disease_cols].head(10)

## 3. NaN por columna (dataset origen no tenía información)

In [ ]:
nan_summary = (
    df[PATHOLOGY_COLS]
    .isna()
    .sum()
    .rename('nan_count')
    .to_frame()
)
nan_summary['pct_of_total'] = (nan_summary['nan_count'] / len(df) * 100).round(2)
nan_summary = nan_summary.sort_values('nan_count', ascending=False)
print('NaN por columna de patología:')
nan_summary

In [ ]:
# NaN breakdown by source dataset
print('NaN por fuente y columna:')
df.groupby('source')[PATHOLOGY_COLS].apply(lambda g: g.isna().sum()).T